<a href="https://colab.research.google.com/github/NMinarecioglu/kizilirmak-drought-propagation/blob/main/CWT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
   pip install pycwt pandas numpy scipy openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 9.0 MB/s eta 0:00:00


In [ ]:
"""
CWT Dominant Period Extractor
Otomatik olarak her istasyon x indis x ölçek kombinasyonu için:
  - Global spektrumdaki anlamlı pik periyotları
  - Bu periyotların aktif olduğu yıl aralıkları
  - XWT/WTC için faz ve koherans özeti
Excel dosyasına yazar.

Requirements:
    pip install pycwt pandas numpy scipy openpyxl
"""

import numpy as np
import pandas as pd
import pycwt as wavelet
from scipy.signal import find_peaks
from scipy.stats import pearsonr
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
STATION_FILES = {
    "Kastamonu" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kastamonu_output_indices",
    "Sivas"     : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Sivas_output_indices",
    "Kayseri"   : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kayseri_output_indices",
    "Yozgat"    : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Yozgat_output_indices",
    "Kirikkale" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kirikkale_output_indices",
}
DATE_COL   = "Date"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/CWT_Tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

INDEX_COLS = ["SPI-3","SPI-6","SPI-12",
              "SDI-3","SDI-6","SDI-12",
              "RDI-3","RDI-6","RDI-12"]

# Wavelet parameters
DT     = 1/12   # monthly data
DJ     = 0.25
S0     = 2 * DT
J      = int(7 / DJ)
MOTHER = wavelet.Morlet(6)
ALPHA  = 0.05
# ============================================================


# ----------------------------------------------------------
# Load station
# ----------------------------------------------------------
def load_station(path, date_col):
    if os.path.isdir(path):
        files = sorted([
            os.path.join(path, f) for f in os.listdir(path)
            if f.endswith((".xlsx",".xls",".csv")) and not f.startswith("~")
        ])
        seen = set(); frames = []
        for fp in files:
            ext = os.path.splitext(fp)[1].lower()
            tmp = pd.read_excel(fp, parse_dates=[date_col]) \
                  if ext in [".xlsx",".xls"] \
                  else pd.read_csv(fp, parse_dates=[date_col])
            new_cols = [date_col]+[c for c in tmp.columns
                                   if c != date_col and c not in seen]
            tmp = tmp[new_cols]; seen.update(tmp.columns)
            frames.append(tmp)
        df = frames[0]
        for f in frames[1:]:
            df = pd.merge(df, f, on=date_col, how="outer")
    else:
        ext = os.path.splitext(path)[1].lower()
        df  = pd.read_excel(path, parse_dates=[date_col]) \
              if ext in [".xlsx",".xls"] \
              else pd.read_csv(path, parse_dates=[date_col])
    return df.sort_values(date_col).reset_index(drop=True)


# ----------------------------------------------------------
# CWT Analysis — extract dominant periods
# ----------------------------------------------------------
def cwt_analysis(series_values, dates):
    """
    Returns:
      dominant_periods : list of significant peak periods (years)
      active_windows   : list of (start_year, end_year) for each peak
      global_power     : array of global wavelet power
      period           : array of periods
      signif_mask      : boolean array where global power > significance
    """
    x   = series_values.astype(float)
    x   = (x - x.mean()) / x.std()
    n   = len(x)

    wave, scales, freqs, coi, _, _ = wavelet.cwt(x, DT, DJ, S0, J, MOTHER)
    power  = np.abs(wave) ** 2
    period = 1.0 / freqs   # years

    # Global wavelet spectrum
    glbl_power = power.mean(axis=1)
    dof        = n - scales
    glbl_signif, _ = wavelet.significance(
        1.0, DT, scales, 1, alpha=0,
        significance_level=1 - ALPHA,
        dof=dof, wavelet=MOTHER
    )
    signif_mask = glbl_power > glbl_signif

    # Find significant peaks in global spectrum
    sig_power = np.where(signif_mask, glbl_power, 0)
    peaks, _  = find_peaks(sig_power, height=0)

    dominant_periods = []
    active_windows   = []

    year_arr = pd.DatetimeIndex(dates).year.values

    for pk in peaks:
        p = period[pk]
        dominant_periods.append(round(p, 1))

        # Find time window where this period is significant
        # (local power > significance level)
        local_signif, _ = wavelet.significance(
            1.0, DT, scales, 0, alpha=0,
            significance_level=1 - ALPHA,
            wavelet=MOTHER
        )
        local_sig_mask = power[pk, :] > local_signif[pk]

        # COI mask — exclude edge effects
        coi_interp = np.interp(
            np.arange(n),
            np.linspace(0, n-1, len(coi)),
            coi
        )
        coi_mask = period[pk] < coi_interp

        valid = local_sig_mask & coi_mask
        if valid.any():
            active_years = year_arr[valid]
            active_windows.append(
                (int(active_years.min()), int(active_years.max()))
            )
        else:
            active_windows.append((np.nan, np.nan))

    return dominant_periods, active_windows, glbl_power, period, signif_mask


# ----------------------------------------------------------
# XWT/WTC Phase Analysis
# ----------------------------------------------------------
def xwt_phase_analysis(x_vals, y_vals):
    """
    Returns mean phase angle (degrees) and mean coherence
    in significant regions.
    Positive phase → X leads Y
    """
    x = (x_vals - x_vals.mean()) / x_vals.std()
    y = (y_vals - y_vals.mean()) / y_vals.std()
    n = min(len(x), len(y))
    x, y = x[:n], y[:n]

    wave_x, scales, freqs, coi, _, _ = wavelet.cwt(x, DT, DJ, S0, J, MOTHER)
    wave_y, _,      _,     _,  _, _ = wavelet.cwt(y, DT, DJ, S0, J, MOTHER)

    # XWT
    xwt   = wave_x * np.conj(wave_y)
    phase = np.angle(xwt, deg=True)   # degrees

    # Significance
    sx, _ = wavelet.significance(1.0, DT, scales, 0, alpha=0,
                                  significance_level=1-ALPHA, wavelet=MOTHER)
    sy, _ = wavelet.significance(1.0, DT, scales, 0, alpha=0,
                                  significance_level=1-ALPHA, wavelet=MOTHER)
    sig95    = np.sqrt(sx[:,None] * sy[:,None])
    sig_mask = (np.abs(xwt) / sig95) > 1

    # WTC (safe)
    try:
        import pycwt.helpers as _h
        _orig = _h.ar1
        def _safe(xx):
            try: return _orig(xx)
            except Warning: return 0.0
        _h.ar1 = _safe
        wct_res = wavelet.wct(x, y, DT, DJ, S0, J,
                              significance_level=1-ALPHA,
                              wavelet=MOTHER, normalize=True, cache=True)
        _h.ar1 = _orig
        if len(wct_res) == 6:
            WTC, aWTC, _, coi_w, freqs_w, signif_w = wct_res
        else:
            WTC, aWTC, _, coi_w, freqs_w = wct_res
            signif_w = np.ones(WTC.shape[0]) * 0.5
        mean_coherence = float(np.abs(WTC[sig_mask[:WTC.shape[0],
                                          :WTC.shape[1]]]).mean()) \
                         if sig_mask.any() else np.nan
    except Exception:
        mean_coherence = np.nan

    # Mean phase in significant XWT regions (months lag)
    if sig_mask.any():
        ph_sig       = phase[sig_mask]
        mean_phase_d = float(np.nanmean(ph_sig))
        # Convert degrees to months lag:
        # phase in degrees / 360 * period_in_months
        # Use median significant period as reference
        sig_periods  = (1.0/freqs)[sig_mask.any(axis=1)]
        ref_period_m = float(np.median(sig_periods)) * 12
        phase_lag_m  = mean_phase_d / 360.0 * ref_period_m
    else:
        mean_phase_d = np.nan
        phase_lag_m  = np.nan

    return mean_phase_d, phase_lag_m, mean_coherence


# ============================================================
# MAIN
# ============================================================
print("="*60)
print("  CWT Dominant Period Extractor")
print("="*60 + "\n")

# Load all station data
STATION_DATA = {}
for station, path in STATION_FILES.items():
    try:
        STATION_DATA[station] = load_station(path, DATE_COL)
        print(f"  Loaded: {station} ({len(STATION_DATA[station])} rows)")
    except Exception as e:
        print(f"  [ERROR] {station}: {e}")

print()

# ── Sheet 1: CWT Summary ─────────────────────────────────
cwt_records = []

for station in STATION_FILES.keys():
    df = STATION_DATA.get(station)
    if df is None:
        continue

    for col in INDEX_COLS:
        if col not in df.columns:
            continue

        idx_name = col.split("-")[0]
        scale    = int(col.split("-")[1])

        series = df[[DATE_COL, col]].dropna()
        if len(series) < 24:
            continue

        dates  = series[DATE_COL].values
        values = series[col].values

        print(f">>> CWT: {station} {col}")
        try:
            dom_periods, act_windows, glbl_power, period, smask = \
                cwt_analysis(values, dates)
        except Exception as e:
            print(f"    [ERROR] {e}")
            continue

        # All significant periods (not just peaks)
        sig_periods = period[smask]
        if len(sig_periods) > 0:
            sig_range = f"{sig_periods.min():.1f}–{sig_periods.max():.1f}"
        else:
            sig_range = "None"

        # Format dominant periods and windows
        if dom_periods:
            dp_str = ", ".join([f"{p:.1f}" for p in dom_periods])
            aw_str = ", ".join([
                f"{w[0]}–{w[1]}" if not np.isnan(w[0]) else "N/A"
                for w in act_windows
            ])
        else:
            dp_str = "None"
            aw_str = "None"

        cwt_records.append({
            "Station"                  : station,
            "Index"                    : idx_name,
            "Scale (month)"            : scale,
            "Significant Period Range (yr)": sig_range,
            "Dominant Periods (yr)"    : dp_str,
            "Active Time Windows"      : aw_str,
            "N_Sig_Periods"            : int(smask.sum()),
            "Global Power Max"         : round(float(glbl_power.max()), 3),
        })

df_cwt = pd.DataFrame(cwt_records)

# ── Sheet 2: XWT/WTC Summary ─────────────────────────────
xwt_records = []

PAIRS = [
    ("SPI", "RDI"), ("SPI", "SDI")
]

for station in STATION_FILES.keys():
    df = STATION_DATA.get(station)
    if df is None:
        continue

    for idx_x, idx_y in PAIRS:
        for scale in [3, 6, 12]:
            col_x = f"{idx_x}-{scale}"
            col_y = f"{idx_y}-{scale}"

            if col_x not in df.columns or col_y not in df.columns:
                continue

            merged = df[[DATE_COL, col_x, col_y]].dropna()
            if len(merged) < 24:
                continue

            x_vals = merged[col_x].values.astype(float)
            y_vals = merged[col_y].values.astype(float)

            print(f">>> XWT/WTC: {station} {col_x} ↔ {col_y}")
            try:
                phase_d, phase_lag_m, coherence = \
                    xwt_phase_analysis(x_vals, y_vals)
            except Exception as e:
                print(f"    [ERROR] {e}")
                phase_d = phase_lag_m = coherence = np.nan

            # Pearson correlation for reference
            try:
                r, p = pearsonr(x_vals, y_vals)
            except Exception:
                r, p = np.nan, np.nan

            # Phase interpretation
            if not np.isnan(phase_lag_m):
                if phase_lag_m > 0:
                    phase_interp = f"{idx_x} leads by {abs(phase_lag_m):.1f} months"
                elif phase_lag_m < 0:
                    phase_interp = f"{idx_y} leads by {abs(phase_lag_m):.1f} months"
                else:
                    phase_interp = "In-phase"
            else:
                phase_interp = "N/A"

            # Coherence level
            if np.isnan(coherence):
                coh_level = "N/A"
            elif coherence >= 0.7:
                coh_level = "High"
            elif coherence >= 0.5:
                coh_level = "Moderate"
            else:
                coh_level = "Low"

            xwt_records.append({
                "Station"            : station,
                "Pair"               : f"{col_x} ↔ {col_y}",
                "Index X"            : idx_x,
                "Index Y"            : idx_y,
                "Scale (month)"      : scale,
                "Pearson r"          : round(r,  3) if not np.isnan(r) else "N/A",
                "Pearson p"          : round(p,  4) if not np.isnan(p) else "N/A",
                "Mean Phase (°)"     : round(phase_d,     1) if not np.isnan(phase_d)     else "N/A",
                "Phase Lag (months)" : round(phase_lag_m, 1) if not np.isnan(phase_lag_m) else "N/A",
                "Phase Interpretation": phase_interp,
                "Mean Coherence"     : round(coherence,   3) if not np.isnan(coherence)   else "N/A",
                "Coherence Level"    : coh_level,
            })

df_xwt = pd.DataFrame(xwt_records)

# ── Save to Excel ─────────────────────────────────────────
out_path = os.path.join(OUTPUT_DIR, "Wavelet_Summary_Table.xlsx")

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df_cwt.to_excel(writer, sheet_name="CWT_Dominant_Periods", index=False)
    df_xwt.to_excel(writer, sheet_name="XWT_WTC_Phase_Coherence", index=False)

    # Auto column width
    for sheet_name, df_s in [("CWT_Dominant_Periods", df_cwt),
                              ("XWT_WTC_Phase_Coherence", df_xwt)]:
        ws = writer.sheets[sheet_name]
        for col_idx, col in enumerate(df_s.columns, start=1):
            max_len = max(
                df_s[col].astype(str).map(len).max(),
                len(col)
            ) + 3
            from openpyxl.utils import get_column_letter
            ws.column_dimensions[get_column_letter(col_idx)].width = \
                min(max_len, 40)

print(f"\nSaved → {out_path}")
print(f"CWT rows : {len(df_cwt)}")
print(f"XWT rows : {len(df_xwt)}")
print("\nDone! Upload Wavelet_Summary_Table.xlsx to get the Results text.")